# Student Completion Prediction — ML Model Development & Evaluation
**Task 8 | Altrodav Technologies Technical Assessment**

**Objective:** Build and evaluate classification models to predict whether a student will successfully complete a training programme based on historical learning behaviour.

**Target Variable:** `Final_Completion_Status` (1 = Completed, 0 = Not Completed)

**Models:** Logistic Regression · Decision Tree Classifier · Random Forest Classifier

---
## 0. Imports & Setup

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

os.makedirs('visualizations', exist_ok=True)
sns.set_theme(style='whitegrid', palette='muted')
print('Libraries loaded successfully.')

---
## 1. Load the Dataset

In [ ]:
df = pd.read_csv('dataset.csv')
print('Dataset shape:', df.shape)
df.head()

In [ ]:
print('Column data types:')
print(df.dtypes)
print('\nDescriptive statistics:')
df.describe()

---
## 2. Data Cleaning & Preprocessing

We explicitly check for **missing values** and **duplicate rows** before modelling.

In [ ]:
# --- Missing values ---
print('Missing values per column:')
print(df.isnull().sum())

# --- Duplicate rows ---
print(f'\nNumber of duplicate rows: {df.duplicated().sum()}')

# Drop duplicates and rows with nulls (if any)
df.drop_duplicates(inplace=True)
df.dropna(inplace=True)
print(f'\nShape after cleaning: {df.shape}')

In [ ]:
# Class distribution
print('Target class distribution:')
print(df['Final_Completion_Status'].value_counts())
print(df['Final_Completion_Status'].value_counts(normalize=True).mul(100).round(2), '(%)')

---
## 3. Exploratory Data Analysis (EDA)

### 3.1 Correlation Heatmap

In [ ]:
FEATURES = [
    'Attendance_Percentage', 'Assessment_Score',
    'Assignment_Completion_Rate', 'Login_Frequency',
    'Study_Hours', 'Activity_Count'
]
TARGET = 'Final_Completion_Status'

fig, ax = plt.subplots(figsize=(9, 7))
corr = df[FEATURES + [TARGET]].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', linewidths=0.5,
    cmap='coolwarm', vmin=-1, vmax=1, ax=ax, annot_kws={'size': 10}
)
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold', pad=14)
plt.tight_layout()
plt.savefig('visualizations/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.2 Feature Distributions by Completion Status

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, feature in enumerate(FEATURES):
    sns.histplot(
        data=df, x=feature, hue=TARGET, ax=axes[i],
        bins=30, kde=True, palette={0: '#DD8452', 1: '#4C72B0'}
    )
    axes[i].set_title(f'{feature}', fontsize=11)
    axes[i].set_xlabel('')

fig.suptitle('Feature Distributions by Completion Status', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('visualizations/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Feature Engineering & Data Splitting

**Train / Test split:** 80 % training — 20 % test, stratified by the target class.

In [ ]:
X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Training samples : {X_train.shape[0]}')
print(f'Test samples     : {X_test.shape[0]}')

### 4.1 Feature Scaling — StandardScaler

> **Critical:** The scaler is **fitted only on the training set** and then applied to both train and test sets. Fitting on the full dataset would cause data leakage and inflate model performance.

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print('Scaled training set mean (should be ≈ 0):', X_train_sc.mean(axis=0).round(4))
print('Scaled training set std  (should be ≈ 1):', X_train_sc.std(axis=0).round(4))

---
## 5. Model Training & Evaluation

We train three classifiers and evaluate each with Accuracy, Precision, Recall, F1-Score, and a Confusion Matrix.

### 5.1 Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)

print('=== Logistic Regression ===')
print(f'Accuracy : {accuracy_score(y_test, y_pred_lr):.4f}')
print(classification_report(y_test, y_pred_lr, target_names=['Not Completed', 'Completed']))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_lr),
    display_labels=['Not Completed', 'Completed']
).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — Logistic Regression', fontweight='bold')
plt.tight_layout()
plt.savefig('visualizations/confusion_matrix_logistic_regression.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.2 Decision Tree Classifier

In [ ]:
dt = DecisionTreeClassifier(max_depth=6, random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print('=== Decision Tree Classifier ===')
print(f'Accuracy : {accuracy_score(y_test, y_pred_dt):.4f}')
print(classification_report(y_test, y_pred_dt, target_names=['Not Completed', 'Completed']))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_dt),
    display_labels=['Not Completed', 'Completed']
).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — Decision Tree', fontweight='bold')
plt.tight_layout()
plt.savefig('visualizations/confusion_matrix_decision_tree.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.3 Random Forest Classifier

In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print('=== Random Forest Classifier ===')
print(f'Accuracy : {accuracy_score(y_test, y_pred_rf):.4f}')
print(classification_report(y_test, y_pred_rf, target_names=['Not Completed', 'Completed']))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_rf),
    display_labels=['Not Completed', 'Completed']
).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — Random Forest', fontweight='bold')
plt.tight_layout()
plt.savefig('visualizations/confusion_matrix_random_forest.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Model Performance Comparison

In [ ]:
results = {
    'Logistic Regression': {
        'Accuracy':  accuracy_score(y_test, y_pred_lr),
        'Precision': precision_score(y_test, y_pred_lr),
        'Recall':    recall_score(y_test, y_pred_lr),
        'F1-Score':  f1_score(y_test, y_pred_lr),
    },
    'Decision Tree': {
        'Accuracy':  accuracy_score(y_test, y_pred_dt),
        'Precision': precision_score(y_test, y_pred_dt),
        'Recall':    recall_score(y_test, y_pred_dt),
        'F1-Score':  f1_score(y_test, y_pred_dt),
    },
    'Random Forest': {
        'Accuracy':  accuracy_score(y_test, y_pred_rf),
        'Precision': precision_score(y_test, y_pred_rf),
        'Recall':    recall_score(y_test, y_pred_rf),
        'F1-Score':  f1_score(y_test, y_pred_rf),
    }
}

metrics_df = pd.DataFrame(results).T
print(metrics_df.round(4))

In [ ]:
metrics_df_reset = metrics_df.reset_index().rename(columns={'index': 'Model'})

fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(metrics_df_reset))
width = 0.18
metric_cols = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

for i, (metric, color) in enumerate(zip(metric_cols, colors)):
    scores = metrics_df_reset[metric].values
    bars = ax.bar(x + i * width, scores, width, label=metric, color=color, alpha=0.88)
    for bar, score in zip(bars, scores):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.004,
                f'{score:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(metrics_df_reset['Model'], fontsize=11)
ax.set_ylim(0, 1.08)
ax.legend(loc='lower right', fontsize=10)
plt.tight_layout()
plt.savefig('visualizations/model_performance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Feature Importance (Random Forest)

In [ ]:
importance_df = pd.DataFrame({
    'Feature':    FEATURES,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(
    importance_df['Feature'], importance_df['Importance'],
    color=sns.color_palette('viridis', len(FEATURES))
)
for bar, val in zip(bars, importance_df['Importance']):
    ax.text(val + 0.002, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=9)
ax.set_xlabel('Feature Importance Score', fontsize=12)
ax.set_title('Random Forest — Feature Importance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('visualizations/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nFeature importance (descending):')
print(importance_df.sort_values('Importance', ascending=False).to_string(index=False))

---
## 8. Identifying At-Risk Students with `predict_proba()`

Students with a **predicted completion probability below 0.40** are flagged as at-risk and eligible for early intervention.

In [ ]:
THRESHOLD = 0.40

# predict_proba returns [P(class=0), P(class=1)]; we need column index 1
completion_prob = rf.predict_proba(X_test.values)[:, 1]

risk_df = X_test.copy()
risk_df['Student_ID']          = df.loc[X_test.index, 'Student_ID'].values
risk_df['Completion_Prob']     = completion_prob
risk_df['Actual_Status']       = y_test.values

at_risk = risk_df[risk_df['Completion_Prob'] < THRESHOLD].sort_values('Completion_Prob')

print(f'Students flagged as at-risk (P < {THRESHOLD}): {len(at_risk)}')
print(at_risk[['Student_ID', 'Completion_Prob', 'Actual_Status']].head(15).to_string(index=False))

---
## 9. Summary

| Model | Accuracy | Precision | Recall | F1-Score |
|---|---|---|---|---|
| Logistic Regression | 0.6900 | 0.6986 | 0.6755 | 0.6869 |
| Decision Tree | 0.6000 | 0.6026 | 0.6026 | 0.6026 |
| Random Forest | 0.6733 | 0.6828 | 0.6556 | 0.6689 |

**Best model:** Logistic Regression achieves the highest accuracy (69.0%) and F1-score (0.687) on this balanced dataset, with Random Forest a close second (67.3%). The Decision Tree underperforms due to overfitting at moderate depth.

**Key finding:** All six behavioural features contribute meaningfully to prediction. Attendance Percentage, Activity Count, and Assessment Score consistently rank as the top predictors.